# Train YOLO11 for the Multi-UAV Disaster Response project

Trains **Model A** (detect: person, vehicle) or **Model B** (segment: building_damaged,
road_blocked, water) on Google Colab's free GPU, then saves the weights to your Google
Drive so you can use them locally (Phase 3).

**Before you start — build the dataset locally and upload it:**
```bash
# on your Mac, from the repo root:
make build-datasets ARGS="--sources visdrone sard --copy-images"   # Model A detect set
cd data/unified && zip -r detect.zip detect                          # ~2.8 GB
# then upload detect.zip to Google Drive, e.g. MyDrive/uav/detect.zip
```
For **Model B**, build the seg set instead (`--sources rescuenet floodnet`), zip `seg`,
and point `DATASET_ZIP` at it. Set **Runtime -> Change runtime type -> GPU** first.


## 1. Check the GPU and install Ultralytics


In [ ]:
!nvidia-smi -L

In [ ]:
%pip install -q ultralytics
import ultralytics; ultralytics.checks()

## 2. Mount Google Drive (where your dataset zip lives, and where weights are saved)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configure the run
Edit these, then run every cell below. Defaults are for **Model A**; the commented values
are for **Model B**. They mirror `configs/perception/model_a.yaml` / `model_b.yaml`.


In [ ]:
MODEL       = 'yolo11s.pt'      # Model B: 'yolo11s-seg.pt'
TASK        = 'detect'         # Model B: 'segment'
DATASET_ZIP = '/content/drive/MyDrive/uav/detect.zip'   # the zip you uploaded
PROJECT_DIR = '/content/drive/MyDrive/uav/runs'         # weights saved here (on Drive)
RUN_NAME    = 'modelA'         # Model B: 'modelB'

EPOCHS   = 100                 # upper bound; stops early via PATIENCE
PATIENCE = 20                  # stop if val metric plateaus for 20 epochs
IMGSZ    = 1280                # Model B: 1024  (tiny objects need large imgsz)
BATCH    = -1                  # -1 = auto-fit to GPU memory
SEED     = 0                   # reproducibility

## 4. Unzip the dataset to Colab's fast local disk and fix the data.yaml path


In [ ]:
import zipfile, os, glob
os.makedirs('/content/data', exist_ok=True)
print('Extracting', DATASET_ZIP, '...')
with zipfile.ZipFile(DATASET_ZIP) as z:
    z.extractall('/content/data')
yml = glob.glob('/content/data/**/data.yaml', recursive=True)[0]
print('Found data.yaml:', yml)

In [ ]:
import yaml
root = os.path.dirname(yml)
with open(yml) as f:
    d = yaml.safe_load(f)
d['path'] = root                      # rewrite the Mac path to the Colab path
with open(yml, 'w') as f:
    yaml.safe_dump(d, f, sort_keys=False)
print('Dataset config now:'); print(d)
for split in ('train', 'val', 'test'):
    p = os.path.join(root, 'images', split)
    if os.path.isdir(p):
        print(f'  {split}: {len(os.listdir(p))} images')

## 5. Train
First run downloads the pretrained `MODEL` weights automatically. Training resumes-safe:
results land in `PROJECT_DIR/RUN_NAME/`.


In [ ]:
from ultralytics import YOLO
model = YOLO(MODEL)
results = model.train(
    data=yml, task=TASK, epochs=EPOCHS, patience=PATIENCE,
    imgsz=IMGSZ, batch=BATCH, seed=SEED,
    project=PROJECT_DIR, name=RUN_NAME, exist_ok=True, plots=True,
)

## 6. Validate on the val split and print the headline metrics


In [ ]:
metrics = model.val()
m = metrics.seg if TASK == 'segment' else metrics.box
print(f'mAP@50-95 : {m.map:.4f}')
print(f'mAP@50    : {m.map50:.4f}')
print('per-class mAP@50-95:', {metrics.names[i]: round(v,4) for i,v in enumerate(m.maps)})

## 7. Locate the trained weights (already saved on Drive)
Download `best.pt` from Drive to your Mac's `weights/` folder; Phase 3 (`detect_cache.py`)
loads it for offline inference.


In [ ]:
best = os.path.join(PROJECT_DIR, RUN_NAME, 'weights', 'best.pt')
print('Best weights:', best, '(exists:', os.path.exists(best), ')')
print('Also on Drive:', os.path.join(PROJECT_DIR, RUN_NAME))

---
**Next steps**
1. Download `best.pt` -> repo `weights/model_a.pt` (or `model_b.pt`).
2. Locally: `uv sync --extra perception` (torch + ultralytics for CPU inference).
3. Report the mAP numbers from cell 6 back into `docs/BUILD_LOG.md`.
4. Repeat this notebook for the other model.
